# Edge AI Real-Time Interference Study - Colab Test & Debugging Workflow

Questo notebook permette di collaudare e fare il debugging dell'intera catena di esperimenti prima di eseguirla sulla scheda fisica (**NVIDIA Jetson AGX Orin**).

> **IMPORTANTE**: Assicurati che il runtime di Colab sia impostato con **GPU** (*Runtime > Cambia tipo di runtime > T4 GPU*).

--- 
## 1. Installazione Dipendenze di Sistema
Installiamo i tool necessari per l'ambiente Linux: `stress-ng` (generatore carichi/interferenza), `sysstat` (che fornisce `vmstat`), `cmake` e compilatore C/C++.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq stress-ng sysstat cmake build-essential openssh-server openssh-client
!pip install -q huggingface_hub datasets
!nvidia-smi

--- 
## 2. Verifica Cartella di Lavoro e File del Progetto
Se hai caricato la cartella del progetto su Google Drive o tramite `git clone`, posizionati nella root.

In [ ]:
# Se usi Google Drive scommenta le due righe sotto:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/EdgeAI_Real_Time_Interference_Performance_Study

# Verifica file presenti
!ls -la setup/

--- 
## 3. Esecuzione del Setup di Sistema (`setup_system.py`)
Scarica i dataset (**MMLU**, **Spider**), il modello **Llama-3.2-1B-Instruct** quantizzato GGUF e compila `llama.cpp` con accelerazione CUDA (`-DGGML_CUDA=ON`).

In [ ]:
!python setup/setup_system.py

--- 
## 4. Test Connessione SSH verso il tuo PC (via Tunnel Pinggy)

Parametri attivi del tunnel Pinggy:
- **Host**: `spinx-87-6-146-154.run.pinggy-free.link`
- **Porta**: `36297`
- **Utente**: `gabri`

Esegui la cella sotto per verificare che Colab riesca a raggiungere il server OpenSSH sul tuo Windows:

In [ ]:
# Smoke-test di connessione SSH verso il tuo PC Windows tramite Pinggy
!ssh -p 36297 -o StrictHostKeyChecking=accept-new gabri@spinx-87-6-146-154.run.pinggy-free.link "echo 'Connessione SSH da Colab al PC Windows riuscita con successo!'"

--- 
## 5. Test Baseline con `run_experiments.py` e Invio CSV via SSH
Eseguiamo 10 task di test della **Baseline** (senza stressor) con `vmstat` e invio automatico dei `.csv` generati direttamente sul tuo PC Windows nella cartella del progetto.

> **Nota:** usiamo `--no-reboot` e `--transient-wait 0` per l'ambiente Colab.

In [ ]:
!python setup/run_experiments.py \
  --model slm \
  --device gpu \
  --benchmark mmlu \
  --no-stressor \
  --samples 10 \
  --repetitions 1 \
  --transient-wait 0 \
  --no-reboot \
  --ssh-target "gabri@spinx-87-6-146-154.run.pinggy-free.link" \
  --ssh-port 36297 \
  --ssh-dest "C:/Users/gabri/OneDrive/Desktop/Universita/Magistrale/Real-Time Systems and Industrial Applications/EdgeAI_Real_Time_Interference_Performance_Study"

Verifichiamo i file locali generati nell'ambiente Colab:

In [ ]:
print("=== File locali generati in Colab ===")
!ls -la data_timeseries/ results/
print("\nControlla ora sul tuo PC Windows: i file CSV devono essere comparsi in data_timeseries/ e results/!")

--- 
## 6. Test Calcolo Dimensionamento Campioni (`compute_sample_size.py`)
Calcoliamo il numero di campioni $n$ per gli stressor usando il Teorema del Limite Centrale (**CLT**), **intervallo di confidenza al 95%** ($z = 1.96$) ed **errore al 5%** sui task appena raccolti:

In [ ]:
!python setup/compute_sample_size.py \
  --model slm \
  --device gpu \
  --benchmark mmlu \
  --confidence 0.95 \
  --error 0.05

--- 
## 7. Test con Stressor (`stress-ng`) e Invio CSV via SSH
Eseguiamo una prova di inferenza con carico di interferenza attivo (stressor `cpu`):
- **Fattori Sperimentali DoE**: esclusivamente il **numero di worker** ($N \in \{25\%, 50\%, 75\%, 100\%\}$) e la **size** ($S$, solo per `mem` e `io`).
- **Durata**: costante e fissa al **tempo totale** ($t_{\text{tot}}$) per l'intera inferenza (copre prefill e decode senza interruzioni).
- Verifica il monitoraggio di `vmstat 1` e il watchdog di timeout per task a 5 minuti (300s).
- Invia automaticamente i CSV time-series e DoE (`results/doe_*.csv`) sul tuo PC Windows.

In [ ]:
!python setup/run_experiments.py \
  --model slm \
  --device gpu \
  --benchmark mmlu \
  --use-stressor \
  --stressor cpu \
  --samples 5 \
  --repetitions 1 \
  --transient-wait 0 \
  --no-reboot \
  --ssh-target "gabri@spinx-87-6-146-154.run.pinggy-free.link" \
  --ssh-port 36297 \
  --ssh-dest "C:/Users/gabri/OneDrive/Desktop/Universita/Magistrale/Real-Time Systems and Industrial Applications/EdgeAI_Real_Time_Interference_Performance_Study"

--- 
## 8. Ispezione Finale dei Risultati e del DoE

In [ ]:
import pandas as pd
import glob

summary_files = glob.glob("results/doe_*.csv")
if summary_files:
    print(f"Contenuto di {summary_files[0]}:")
    display(pd.read_csv(summary_files[0]))
else:
    print("Nessun file DoE generato.")

if glob.glob("data_timeseries/*timeseries.csv"):
    ts_sample = glob.glob("data_timeseries/*timeseries.csv")[0]
    print(f"\nAnteprima Time-Series vmstat ({ts_sample}):")
    display(pd.read_csv(ts_sample).head(5))